# COMP3710 Lab 1 - Part 3 CUDA evidence

This notebook runs the existing reviewed Sierpinski gasket project on CUDA. It does **not** replace the source files. Run each cell separately and read the result before continuing.

## Step 0 - Enable CUDA

Choose **Runtime > Change runtime type > T4 GPU**, save, and then run the next cell. Colab may assign a different NVIDIA GPU model.

In [ ]:
from pathlib import Path
import io
import os
import platform
import shutil
import subprocess
import sys
import zipfile

import torch

PROJECT_DIR = Path('/content/comp3710-part3')
RUN_LOGS = []

def run_step(label, command):
    completed = subprocess.run(
        command, text=True, capture_output=True, cwd=PROJECT_DIR
    )
    block = (
        f'\n===== {label} =====\n'
        f"Command: {' '.join(map(str, command))}\n"
        f'Return code: {completed.returncode}\n'
        f'--- stdout ---\n{completed.stdout}'
        f'--- stderr ---\n{completed.stderr}'
    )
    RUN_LOGS.append(block)
    print(block)
    if completed.returncode != 0:
        raise RuntimeError(f'{label} failed; read the output above')
    return completed

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU and reconnect.'
    )

smi = subprocess.run(['nvidia-smi'], text=True, capture_output=True)
device_report = (
    '===== Colab CUDA environment =====\n'
    f'Python: {platform.python_version()}\n'
    f'PyTorch: {torch.__version__}\n'
    f'CUDA available: {torch.cuda.is_available()}\n'
    f'PyTorch CUDA runtime: {torch.version.cuda}\n'
    f'GPU: {torch.cuda.get_device_name(0)}\n'
    f'\n===== nvidia-smi =====\n{smi.stdout}{smi.stderr}'
)
RUN_LOGS.append(device_report)
print(device_report)

## Step 1 - Upload the reviewed project bundle

Upload `dist/COMP3710_Part3_Colab_Bundle.zip` from the local project. The bundle contains the same source and verification files already tested on the Mac.

In [ ]:
from google.colab import files

uploaded = files.upload()
bundle_names = [name for name in uploaded if name.endswith('.zip')]
if len(bundle_names) != 1:
    raise FileNotFoundError('Upload exactly one COMP3710 Part 3 bundle ZIP')

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(uploaded[bundle_names[0]])) as archive:
    archive.extractall(PROJECT_DIR)

OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'colab'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
expected = {
    'src/reference_gasket.py',
    'src/torch_gasket.py',
    'src/analyse_gasket.py',
    'src/benchmark_gasket.py',
    'tests/verify_reference.py',
    'tests/verify_torch_gasket.py',
    'tests/verify_analysis.py',
    'tests/verify_benchmark.py',
}
missing = sorted(path for path in expected if not (PROJECT_DIR / path).is_file())
if missing:
    raise FileNotFoundError(f'Bundle is missing required files: {missing}')

print(f'Project ready: {PROJECT_DIR}')
for path in sorted(expected):
    print(f'Ready: {path}')

## Step 2 - Run every independent check

The PyTorch verification should print both `Verified device: cpu` and `Verified device: cuda`.

In [ ]:
for label, script in [
    ('Reference checks', 'tests/verify_reference.py'),
    ('PyTorch CPU and CUDA checks', 'tests/verify_torch_gasket.py'),
    ('Dimension-analysis checks', 'tests/verify_analysis.py'),
    ('Benchmark-runner checks', 'tests/verify_benchmark.py'),
]:
    run_step(label, [sys.executable, script])

## Step 3 - Generate the reviewed gasket explicitly on CUDA

In [ ]:
cuda_image = OUTPUT_DIR / 'torch_gasket_cuda.png'
run_step(
    'Reviewed PyTorch gasket on CUDA',
    [
        sys.executable, 'src/torch_gasket.py',
        '--rows', '1024', '--device', 'cuda',
        '--output', str(cuda_image),
    ],
)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(cuda_image), width=1000))

## Step 4 - Repeat the dimension and colour analysis from a CUDA-generated mask

In [ ]:
analysis_image = OUTPUT_DIR / 'gasket_analysis_cuda.png'
analysis_csv = OUTPUT_DIR / 'box_counts_cuda.csv'
run_step(
    'Box-counting analysis from CUDA mask',
    [
        sys.executable, 'src/analyse_gasket.py',
        '--rows', '1024', '--device', 'cuda',
        '--output', str(analysis_image),
        '--csv', str(analysis_csv),
    ],
)
display(Image(filename=str(analysis_image), width=1400))

## Step 5 - Compare Colab CPU and CUDA fairly

Each device/size pair uses three discarded warm-ups and nine measured, synchronised runs.

In [ ]:
benchmark_image = OUTPUT_DIR / 'benchmark_cpu_cuda.png'
benchmark_csv = OUTPUT_DIR / 'benchmark_cpu_cuda.csv'
run_step(
    'Colab CPU and CUDA benchmark',
    [
        sys.executable, 'src/benchmark_gasket.py',
        '--sizes', '256', '512', '1024', '2048', '4096',
        '--devices', 'cpu', 'cuda',
        '--warmups', '3', '--repeats', '9',
        '--output', str(benchmark_image),
        '--csv', str(benchmark_csv),
    ],
)
display(Image(filename=str(benchmark_image), width=1100))

## Step 6 - Download the CUDA evidence package

Download the ZIP before the Colab runtime disconnects.

In [ ]:
evidence_log = OUTPUT_DIR / 'part3_colab_cuda_evidence.txt'
evidence_log.write_text('\n'.join(RUN_LOGS), encoding='utf-8')

package_path = Path('/content/COMP3710_Part3_Colab_Evidence.zip')
with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ('src', 'tests'):
        for path in sorted((PROJECT_DIR / folder).rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                archive.write(path, arcname=str(path.relative_to(PROJECT_DIR)))
    for name in ('README.md', 'PART3_NOTES.md', 'BENCHMARK_RESULTS.md', 'REFERENCES.md'):
        path = PROJECT_DIR / name
        if path.is_file():
            archive.write(path, arcname=name)
    for path in sorted(OUTPUT_DIR.glob('*')):
        if path.is_file():
            archive.write(path, arcname=f'outputs/colab/{path.name}')

print(f'Evidence package ready: {package_path} ({package_path.stat().st_size} bytes)')
files.download(str(package_path))

## Final checklist

Before finishing, confirm that you can explain:

- why `c & (row-c) == 0` identifies odd Pascal entries;
- how `(N,1)` and `(1,N)` tensors broadcast to an `(N,N)` calculation;
- why the final image alone is moved to CPU;
- how box counts estimate `log(3)/log(2)`; and
- why CPU, MPS, and CUDA timings must be measured rather than assumed.